In [7]:
import re
from jinja2 import Environment
from pathlib import Path
from collections import defaultdict

atomic_masses = {
    "H": 1.0079, "C": 12.011, "N": 14.0067, "O": 15.9994,
    "F": 18.9984, "Cl": 35.45, "S": 32.06, "P": 30.97,
    "Br": 79.904, "I": 126.90, "Zn": 65.38, "Cu": 63.546
}


def generate_angles(bonds):
    # bonds: list of (atom1, atom2)
    neighbors = defaultdict(set)
    for a, b in bonds:
        neighbors[a].add(b)
        neighbors[b].add(a)
    angles = set()
    for center in neighbors:
        nbs = list(neighbors[center])
        for i in range(len(nbs)):
            for j in range(i+1, len(nbs)):
                angles.add((nbs[i], center, nbs[j]))
    return list(angles)

def generate_dihedrals(bonds):
    # bonds: list of (atom1, atom2)
    neighbors = defaultdict(set)
    for a, b in bonds:
        neighbors[a].add(b)
        neighbors[b].add(a)
    dihedrals = set()
    # For all bonds (b-c), find (a-b-c-d) where a!=c, d!=b
    for b in neighbors:
        for c in neighbors[b]:
            for a in neighbors[b]:
                if a == c:
                    continue
                for d in neighbors[c]:
                    if d == b:
                        continue
                    # To prevent duplicate: sort (a,b,c,d)
                    dihed = (a, b, c, d)
                    dihedrals.add(dihed)
    # Remove duplicates (could do more refinement for directionality if needed)
    return list(dihedrals)

def generate_impropers(bonds):
    neighbors = defaultdict(set)
    for a, b in bonds:
        neighbors[a].add(b)
        neighbors[b].add(a)
    impropers = set()
    for center, nbs in neighbors.items():
        nbs = list(nbs)
        if len(nbs) >= 3:
            # All unique 3-combinations from neighbors
            for i in range(len(nbs)):
                for j in range(i+1, len(nbs)):
                    for k in range(j+1, len(nbs)):
                        improper = (center, nbs[i], nbs[j], nbs[k])
                        impropers.add(improper)
    return list(impropers)

def parse_cgenff_rtf(rtf_file, molecule):
    """
    Only parse the specified RESI <molecule> block from a CHARMM RTF file.
    Returns atoms, types, charges, bonds, angles, dihedrals, impropers
    """
    atoms, types, charges, bonds = [], [], {}, []
    angles, dihedrals, impropers = [], [], []
    in_resi = False
    with open(rtf_file) as f:
        for line in f:
            l = line.strip()
            if l.startswith(f"RESI {molecule}"):
                in_resi = True
                continue
            # New RESI block starts: break
            if in_resi and l.startswith("RESI "):
                break
            if in_resi:
                if l.startswith("ATOM"):
                    parts = l.split()
                    if len(parts) >= 4:
                        name, ff_type, charge = parts[1:4]
                        atoms.append((name, ff_type))
                        types.append(ff_type)
                        charges[name] = float(charge)
                elif l.startswith("BOND"):
                    parts = l.split()[1:]
                    bonds.extend([(parts[j], parts[j+1]) for j in range(0, len(parts), 2)])
                elif l.startswith("ANGLE"):
                    parts = l.split()[1:]
                    angles.extend([(parts[j], parts[j+1], parts[j+2]) for j in range(0, len(parts), 3)])
                elif l.startswith("DIHE") or l.startswith("DIHEDRAL"):
                    parts = l.split()[1:]
                    dihedrals.extend([(parts[j], parts[j+1], parts[j+2], parts[j+3]) for j in range(0, len(parts), 4)])
                elif l.startswith("IMPH") or l.startswith("IMPROPER"):
                    parts = l.split()[1:]
                    impropers.extend([(parts[j], parts[j+1], parts[j+2], parts[j+3]) for j in range(0, len(parts), 4)])
    
    if not angles:
        angles = generate_angles(bonds)
    if not dihedrals:
        dihedrals = generate_dihedrals(bonds)
    if not impropers:
        impropers = generate_impropers(bonds)
    return atoms, types, charges, bonds, angles, dihedrals, impropers

def is_float(val):
    try:
        float(val)
        return True
    except ValueError:
        return False

def parse_cgenff_prm(prm_file):
    masses = {}
    lj = {}
    bonds = {}
    angles_2 = {}
    angles_4 = {}
    dihedrals = {}
    impropers = {}

    section = None
    with open(prm_file) as f:
        for line in f:
            l = line.strip()

            # Skip empty lines and comments
            if not l or l.startswith("!"):
                continue

            # Section header detection
            if l.startswith("MASS"):
                section = "MASS"
            elif l.startswith("NONBONDED"):
                section = "NONBONDED"
            elif l.startswith("BONDS"):
                section = "BONDS"
            elif l.startswith("ANGLES"):
                section = "ANGLES"
            elif l.startswith("DIHEDRALS"):
                section = "DIHEDRALS"
            elif l.startswith("IMPROPERS"):
                section = "IMPROPERS"
            elif l.startswith("END"):
                section = None
                continue

            # Parse the corresponding section
            if section == "MASS" and l.startswith("MASS"):
                # MASS   258 HGA3     1.00800  ! description
                parts = l.split()
                if len(parts) >= 4:
                    atype = parts[2]
                    mass = float(parts[3])
                    masses[atype] = mass

            elif section == "NONBONDED":
                # HGA3     0.0       -0.0240     1.3400 ! ...
                # Skip header lines containing non-numeric entries
                parts = l.split()
                # Ignore if line is a sub-header or not the correct format
                if (
                    len(parts) >= 4
                    and parts[0][0].isalpha()
                    and all(x.replace('.', '', 1).replace('-', '', 1).isdigit() for x in [parts[2], parts[3]])
                ):
                    atype = parts[0]
                    epsilon = float(parts[2])
                    rmin2 = float(parts[3])
                    lj[atype] = (epsilon, rmin2)

            elif section == "BONDS":
                # CG321  NG321  460.0   1.3400 ! ...
                parts = l.split()
                if len(parts) >= 4:
                    a1, a2, k, r0 = parts[:4]
                    bonds[(a1, a2)] = (float(k), float(r0))

            elif section == "ANGLES":
                # HGA3  CG331  HGA3     35.50    108.40    5.40   1.80200 ! ...
                # or   HGA3  CG331  HGA3     35.50    108.40
                parts = l.split()
                if len(parts) >= 5:
                    a1, a2, a3 = parts[:3]
                    k = float(parts[3])
                    theta = float(parts[4])
                    if len(parts) > 6 and is_float(parts[5]) and is_float(parts[6]):
                        Kub = float(parts[5])
                        S0 = float(parts[6])
                        angles_4[(a1, a2, a3)] = (k, theta, Kub, S0)
                    else:
                        angles_2[(a1, a2, a3)] = (k, theta)


            elif section == "DIHEDRALS":
                # e.g. CG2R53 CG251O CG25C1 CG2R53     6.4000  2   180.00
                # Handles multiple terms per line
                line_no_comment = l.split("!")[0].strip()
                parts = line_no_comment.split()
                if len(parts) >= 7:
                    a1, a2, a3, a4 = parts[:4]
                    params = parts[4:]
                    n_terms = len(params) // 3
                    for i in range(n_terms):
                        k = float(params[3 * i])
                        n = int(float(params[3 * i + 1]))
                        phi = float(params[3 * i + 2])
                        key = (a1, a2, a3, a4, n)
                        dihedrals[key] = (k, phi)

            elif section == "IMPROPERS":
                # e.g. CG2D1  CG331  NG2D1  HGA4      25.0000  0     0.00
                line_no_comment = l.split("!")[0].strip()
                parts = line_no_comment.split()
                if len(parts) >= 7:
                    a1, a2, a3, a4 = parts[:4]
                    k = float(parts[4])
                    n = int(float(parts[5]))
                    phi = float(parts[6])
                    key = (a1, a2, a3, a4)
                    impropers[key] = (k, n, phi)
    return masses, lj, bonds, angles_2, angles_4, dihedrals, impropers

def read_xyz(xyz_file):
    coords = []
    with open(xyz_file, 'r') as f:
        lines = f.readlines()[2:]  # skip header
        for line in lines:
            if not line.strip():
                continue
            _, x, y, z = line.strip().split()
            coords.append((float(x), float(y), float(z)))
    return coords


In [11]:
def generate_lt(molecule, xyz_file, rtf_file, prm_file, output_file):
    # Parse CHARMM RTF
    atoms, types, charges, bonds, angles, dihedrals, impropers = parse_cgenff_rtf(rtf_file, molecule)
    atom_name_to_type = dict(atoms)

    # force field parameter extraction
    masses, lj_params, bond_params, angle_2_params, angle_4_params, dihedral_params, improper_params = parse_cgenff_prm(prm_file)

    atom_types_used = set(types)
    bond_types_used = set(tuple(sorted([atom_name_to_type[a1], atom_name_to_type[a2]])) for (a1, a2) in bonds)
    angle_types_used = set((atom_name_to_type[a1], atom_name_to_type[a2], atom_name_to_type[a3]) for (a1,a2,a3) in angles)
    dihedral_types_used = set((atom_name_to_type[a1], atom_name_to_type[a2], atom_name_to_type[a3], atom_name_to_type[a4]) for (a1,a2,a3,a4) in dihedrals)
    improper_types_used = set((atom_name_to_type[a1], atom_name_to_type[a2], atom_name_to_type[a3], atom_name_to_type[a4]) for (a1,a2,a3,a4) in impropers)

    filtered_masses = {k: v for k, v in masses.items() if k in atom_types_used}
    filtered_lj     = {k: v for k, v in lj_params.items() if k in atom_types_used}
    filtered_bonds  = {k: v for k, v in bond_params.items() if set(k).issubset(bond_types_used)}
    filtered_angles_2 = {k: v for k, v in angle_2_params.items() if k in angle_types_used or k[::-1] in angle_types_used}
    filtered_angles_4 = {k: v for k, v in angle_4_params.items() if k in angle_types_used or k[::-1] in angle_types_used}
    filtered_dihedrals = {k: v for k, v in dihedral_params.items() if k in dihedral_types_used or k[::-1] in dihedral_types_used}
    filtered_impropers = {k: v for k, v in improper_params.items() if k in improper_types_used or k[::-1] in improper_types_used}

    coords = read_xyz(xyz_file)

    template_str = """
guest {

  write_once("In Init") {
    atom_style full
    pair_style lj/cut/coul/long 10.0
    kspace_style pppm 1.0e-4
    bond_style harmonic
    angle_style hybrid harmonic charmm
    dihedral_style charmm
    improper_style harmonic
    special_bonds amber
  }

  write_once("In Settings") {
    {% for at, (eps, rmin) in lj_params.items() %}
    pair_coeff @atom:{{ at }} @atom:{{ at }} {{ "%.4f"|format(eps) }} {{ "%.4f"|format(rmin) }}
    {% endfor %}
    {% for (a, b), (k, r0) in bond_params.items() %}
    bond_coeff @bond:{{ a }}_{{ b }} {{ "%.1f"|format(k) }} {{ "%.4f"|format(r0) }}
    {% endfor %}
    {# Harmonic angle only (2 parameters) #}
    {% for (a1, a2, a3), (k, theta) in angle_2_params.items() %}
    angle_coeff @angle:{{ a1 }}_{{ a2 }}_{{ a3 }} harmonic {{ "%.1f"|format(k) }} {{ "%.1f"|format(theta) }}
    {% endfor %}
    {# Harmonic+UB angle (4 parameters) #}
    {% for (a1, a2, a3), (k, theta, Kub, S0) in angle_4_params.items() %}
    angle_coeff @angle:{{ a1 }}_{{ a2 }}_{{ a3 }} charmm {{ "%.1f"|format(k) }} {{ "%.1f"|format(theta) }} {{ "%.2f"|format(Kub) }} {{ "%.3f"|format(S0) }}
    {% endfor %}
    {% for (a1, a2, a3, a4, n), (k, phi) in dihedral_params.items() %}
    dihedral_coeff @dihedral:{{ a1 }}_{{ a2 }}_{{ a3 }}_{{ a4 }} {{ "%.4f"|format(k) }} {{ n }} {{ "%.1f"|format(phi) }} 1.0
    {% endfor %}
    {% for (a1, a2, a3, a4), (k, n, phi) in improper_params.items() %}
    improper_coeff @improper:{{ a1 }}_{{ a2 }}_{{ a3 }}_{{ a4 }} {{ "%.4f"|format(k) }} {{ "%.1f"|format(phi) }}
    {% endfor %}
  }

  write_once("Data Masses") {
    {% for at, m in masses.items() %}
    @atom:{{ at }} {{ "%.4f"|format(m) }}
    {% endfor %}
  }

  write("Data Atoms") {
    {% for i, (name, ff_type) in enumerate(atoms) %}
    $atom:{{ name }} $mol:{{ molecule }} @atom:{{ ff_type }} {{ charges[name] }} {{ coords[i][0] }} {{ coords[i][1] }} {{ coords[i][2] }}
    {% endfor %}
  }

  write("Data Bonds") {
    {% for i, (a1, a2) in enumerate(bonds, 1) %}
    $bond:b{{ i }} @bond:{{ atom_name_to_type[a1] }}_{{ atom_name_to_type[a2] }} $atom:{{ a1 }} $atom:{{ a2 }}
    {% endfor %}
  }

  write("Data Angles") {
      {% for i, (a1, a2, a3) in enumerate(angles, 1) %}
          {% set ttype = (atom_name_to_type[a1], atom_name_to_type[a2], atom_name_to_type[a3]) %}
          {% if ttype in angle_2_params or ttype[::-1] in angle_2_params or ttype in angle_4_params or ttype[::-1] in angle_4_params %}
      $angle:a{{ i }} @angle:{{ ttype[0] }}_{{ ttype[1] }}_{{ ttype[2] }} $atom:{{ a1 }} $atom:{{ a2 }} $atom:{{ a3 }}
          {% endif %}
      {% endfor %}
  }

  write("Data Dihedrals") {
      {% for i, (a1, a2, a3, a4) in enumerate(dihedrals, 1) %}
          {% set ttype = (atom_name_to_type[a1], atom_name_to_type[a2], atom_name_to_type[a3], atom_name_to_type[a4]) %}
          {% if ttype in dihedral_params or ttype[::-1] in dihedral_params %}
      $dihedral:d{{ i }} @dihedral:{{ ttype[0] }}_{{ ttype[1] }}_{{ ttype[2] }}_{{ ttype[3] }} $atom:{{ a1 }} $atom:{{ a2 }} $atom:{{ a3 }} $atom:{{ a4 }}
          {% endif %}
      {% endfor %}
  }

  write("Data Impropers") {
      {% for i, (a1, a2, a3, a4) in enumerate(impropers, 1) %}
          {% set ttype = (atom_name_to_type[a1], atom_name_to_type[a2], atom_name_to_type[a3], atom_name_to_type[a4]) %}
          {% if ttype in improper_params or ttype[::-1] in improper_params %}
      $improper:im{{ i }} @improper:{{ ttype[0] }}_{{ ttype[1] }}_{{ ttype[2] }}_{{ ttype[3] }} $atom:{{ a1 }} $atom:{{ a2 }} $atom:{{ a3 }} $atom:{{ a4 }}
          {% endif %}
      {% endfor %}
  }
}
"""
    env = Environment(trim_blocks=True, lstrip_blocks=True)
    tmpl = env.from_string(template_str)
    rendered = tmpl.render(
        molecule=molecule,
        atoms=atoms,
        atom_name_to_type=atom_name_to_type,
        charges=charges,
        lj_params=filtered_lj,
        bond_params=filtered_bonds,
        angle_2_params=filtered_angles_2,
        angle_4_params=filtered_angles_4,
        dihedral_params=filtered_dihedrals,
        improper_params=filtered_impropers,
        masses=filtered_masses,
        coords=coords,
        bonds=bonds,
        angles=angles,
        dihedrals=dihedrals,
        impropers=impropers,
        enumerate=enumerate
    )
    with open(output_file, 'w') as f:
        f.write(rendered)

generate_lt("MEOH", "methanol.xyz", "/home/users/taeun8991/lammps_auto/Forcefields/CHARMM/cgenff3.0.1/top_all36_cgenff.rtf", "/home/users/taeun8991/lammps_auto/Forcefields/CHARMM/cgenff3.0.1/par_all36_cgenff.prm", "methanol.lt")
